In [1]:
from pyspark.sql import SparkSession
from delta import *
import os

In [2]:
def create_spark_session():
    """
    Creates a SparkSession with Delta Lake and S3/MinIO configurations.
    """
    builder = SparkSession.builder \
        .appName("EnterpriseLakehouseInit") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
        .config("spark.sql.warehouse.dir", "s3a://warehouse/")

    # Add required Maven packages for Delta + AWS S3
    # Note: Versions must be compatible with the Spark version in the Dockerfile (3.5.0)
    return configure_spark_with_delta_pip(builder, extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ]).getOrCreate()

In [3]:
spark = create_spark_session()

In [4]:
def write_table(spark, table_name, df, write_mode="overwrite"):
    table_path = f"s3a://warehouse/{table_name}"
    print(f"Writing Delta table to {table_path}...")
    df.write.format("delta").mode(write_mode).save(table_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA LOCATION '{table_path}'")
    print(f"Table {table_name} created successfully.")

In [5]:
def print_table(spark, table_name):
    print("\n--- Verifying Data via SQL ---")
    spark.sql(f"SELECT * FROM {table_name}").show()

def print_delta_history(spark, table_name):
    print("\n--- Verifying Delta History ---")
    from delta.tables import DeltaTable
    dt = DeltaTable.forPath(spark, f"s3a://warehouse/{table_name}")
    dt.history().show(truncate=False)


In [6]:
data = [
    (1, "Alice", "Engineering", "2023-01-01"),
    (2, "Bob", "Sales", "2023-01-02"),
    (3, "Charlie", "Marketing", "2023-01-03")
]
columns = ["id", "name", "department", "hire_date"]
df = spark.createDataFrame(data, columns)

write_table(spark, "test1", df)
print_table(spark, "test1")
print_delta_history(spark, "test1")



Writing Delta table to s3a://warehouse/test1...
Table test1 created successfully.

--- Verifying Data via SQL ---
+---+-------+-----------+----------+
| id|   name| department| hire_date|
+---+-------+-----------+----------+
|  1|  Alice|Engineering|2023-01-01|
|  3|Charlie|  Marketing|2023-01-03|
|  2|    Bob|      Sales|2023-01-02|
+---+-------+-----------+----------+


--- Verifying Delta History ---
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+